In [66]:
import pandas as pd
from pathlib import Path
import re

In [ ]:
df_stats = pd.read_csv('../data/raw/brackets/brackets-2025.csv')

def determine_round(row):
    # I dislike super python-y statements like this
    multiplier = 4 if row['bracket_type'] == 'team16' else 1
    
    # multiplier is 4 in the regional brackets because there are 4 of them
    # theres only 1 final 4 bracket
    return row['games_in_round'] * 2 * multiplier

df_stats['round_of'] = df_stats.apply(determine_round, axis=1)
df_stats = df_stats.drop(columns=['bracket_type', 'games_in_round'])

df_stats.head(63)

,winning_team_seed,winning_team_name,winning_team_score,losing_team_seed,losing_team_name,losing_team_score,round_of
0,1,Duke,93,16,Mount St. Mary's,49,64
1,9,Baylor,75,8,Mississippi State,72,64
2,5,Oregon,81,12,Liberty,52,64
3,4,Arizona,93,13,Akron,65,64
4,6,BYU,80,11,VCU,71,64
...,...,...,...,...,...,...,...
58,3,Texas Tech,85,10,Arkansas,83,16
59,1,Florida,84,3,Texas Tech,79,8
60,1,Florida,79,1,Auburn,73,4
61,1,Houston,70,1,Duke,67,4


In [ ]:
df_stats = pd.read_csv('../data/raw/teams/2021/alabama/roster.csv')
df_stats.head()

,Player,#,Class,Pos,Height,Weight,Hometown,High School,RSCI Top 100,Summary
0,Jaden Shackelford,5,SO,G,6-3,200,"Hesperia, CA",Hesperia HS,98 (2019),"14.0 Pts, 3.8 Reb, 2.0 Ast"
1,John Petty,23,SR,G,6-5,184,"Huntsville, AL",J.O. Johnson High School,28 (2017),"12.6 Pts, 5.2 Reb, 1.9 Ast"
2,Jahvon Quinerly,13,SO,G,6-1,175,"Hackensack, NJ",Hudson Catholic HS,29 (2018),"12.9 Pts, 2.2 Reb, 3.2 Ast"
3,Herb Jones,1,SR,F,6-8,210,"Greensboro, AL",Hale County High School,NaN,"11.2 Pts, 6.6 Reb, 3.3 Ast"
4,Joshua Primo,11,FR,G,6-6,190,"Toronto, Canada",Huntington Prep HS,80 (2020),"8.1 Pts, 3.4 Reb, 0.8 Ast"


In [98]:
team_data_directory = Path('../data/raw/teams/2021')

# Each data frame
df_stats = pd.DataFrame()
df_info  = pd.DataFrame()

# Loop each team
for team in team_data_directory.iterdir():
    if team.is_dir():
        # File I want
        csv_file = team / 'season_total_per_game.csv'
        if csv_file.exists():
            # Load CSV
            df_temp = pd.read_csv(csv_file)
            # Add column with team name
            df_temp['team'] = team.name
            # Append
            df_stats = pd.concat([df_stats, df_temp])

        csv_file = team / 'team-info.csv'
        if csv_file.exists():
            # Load CSV
            df_temp = pd.read_csv(csv_file)
            # Add column with team name
            df_temp['team'] = team.name
            # Append
            df_info = pd.concat([df_info, df_temp])

df_stats.loc[df_stats.index == 0, 'Type'] = "Team Stat"
df_stats.loc[df_stats.index == 1, 'Type'] = "Team Rank"
df_stats.loc[df_stats.index == 2, 'Type'] = "Opponent Stat"
df_stats.loc[df_stats.index == 3, 'Type'] = "Opponent Rank"
df_stats = df_stats.reset_index()
df_stats.drop(columns=['index', 'Unnamed: 0']).head()

# TODO: Fix the crawler for the Records column
df_info = df_info.drop(columns=['Records'])

# Extract SOS, SRS, ORtg, and DRtg
df_info['SOS']  = df_info['SOS'].str.strip().str.extract(r'^([^ (]*)', expand=False)
df_info['SRS']  = df_info['SRS'].str.strip().str.extract(r'^([^ (]*)', expand=False)
df_info['ORtg'] = df_info['ORtg'].str.strip().str.extract(r'(\d+\.\d+)', expand=False)
df_info['DRtg'] = df_info['DRtg'].str.strip().str.extract(r'(\d+\.\d+)', expand=False)
df_info.head()
# df_info['SOS'].head()

,SOS,SRS,ORtg,DRtg,team
0,10.87,12.37,103.3,101.2,missouri
0,-3.15,4.03,109.0,98.0,eastern-washington
0,9.52,14.79,104.8,97.4,north-carolina
0,10.36,10.13,101.0,101.3,georgetown
0,5.92,27.20,121.5,93.1,gonzaga
